In [ ]:
import warnings
warnings.simplefilter(action='ignore')
import pycisTopic
pycisTopic.__version__

In [ ]:
projDir = '/home/spalo/lu2025-12-50/spalo/scenicplus_storage/'

outDir = projDir + 'pycistopic_output/'
import os
if not os.path.exists(outDir):
    os.makedirs(outDir)

tmpDir = '/home/spalo/lu2025-12-50/spalo/scenicplus_storage/tmp/'

In [ ]:
from pycisTopic.cistopic_class import *
count_matrix1 = pd.read_csv(projDir + 'merged/merged_ATAC_rawc_1.tsv', sep = '\t')
count_matrix2 = pd.read_csv(projDir + 'merged/merged_ATAC_rawc_2.tsv', sep = '\t')
path_to_blacklist = projDir + 'fasta/hg38-blacklist.v2.bed' #downloaded from github.com/Boyle-Lab/Blacklist/blob/master/lists/

In [ ]:
# counts matrix divided in half - check that they have the same rows but different columns
print(count_matrix1.shape)
print(count_matrix2.shape)
print(count_matrix1.index[0:3])
print(count_matrix2.index[0:3])
print(count_matrix1.columns[0:3])
print(count_matrix2.columns[0:3])

In [ ]:
count_matrix = pd.concat([count_matrix1, count_matrix2], axis=1)
print(count_matrix.shape)
del(count_matrix1)
del(count_matrix2)

In [ ]:
# rename rows to be compatible with scenic 
count_matrix.index = count_matrix.index.str.replace("[-]", ":", 1, regex = True)
count_matrix.index

In [ ]:
# Create cisTopic object
cistopic_obj = create_cistopic_object(fragment_matrix = count_matrix, path_to_blacklist = path_to_blacklist)

In [ ]:
#read metadata and change cell names to match counts matrix
cell_data = pd.read_csv(projDir + 'merged/merged_ATAC_md.tsv', sep = '\t')
cell_data.index = cell_data.index.str.replace("[-]", ".", regex = True)
# change seurat cluster names to string so they won't plot in viridis scale
cell_data["seurat_clusters"] = cell_data["seurat_clusters"].astype(str)

cistopic_obj.add_cell_data(cell_data)

In [ ]:
cell_data

In [ ]:
print(cistopic_obj)

In [ ]:
import os
path_to_mallet_binary = '../Mallet/bin/mallet'
# Run models
from pycisTopic.lda_models import *
models = run_cgs_models_mallet(mallet_path = path_to_mallet_binary,
                    cistopic_obj = cistopic_obj,
                    n_topics=[10,20,30,40,45,50,55,60,65,70,75],
                    n_cpu = 48,
                    n_iter = 500,
                    random_state = 555,
                    alpha = 50,
                    alpha_by_topic = True,
                    eta = 0.1,
                    eta_by_topic = False,
                    tmp_path = tmpDir, #Use SCRATCH if many models or big data set
                    save_path = None)

In [ ]:
# Save
import pickle 
with open(outDir + 'merged_Mallet_models_500.pkl', 'wb') as f:
  pickle.dump(models, f)

In [ ]:
#os.mkdir(outDir+'models')
import matplotlib
model = evaluate_models(models,
                     select_model = None, 
                     return_model = True, 
                     metrics = ['Arun_2010','Cao_Juan_2009', 'Minmo_2011', 'loglikelihood'],
                     plot_metrics = False,
                     save = outDir + 'models/merged_model_selection.pdf')

In [ ]:
# Add model to cisTopicObject
cistopic_obj.add_LDA_model(model)
# Save
with open(outDir + 'merged_CT_object.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

In [ ]:
# Load cisTopic object
import pickle
infile = open(outDir + 'merged_CT_object.pkl', 'rb')
cistopic_obj = pickle.load(infile)
infile.close()

In [ ]:
from pycisTopic.clust_vis import *
find_clusters(cistopic_obj,
                 target  = 'cell',
                 k = 10,
                 res = [0.6], 
                 prefix = 'pycisTopic_', 
                 scale = True,
                 split_pattern = '-')

In [ ]:
run_umap(cistopic_obj, target  = 'cell', scale=True)
run_tsne(cistopic_obj, target  = 'cell', scale=True)

In [ ]:
#os.mkdir(outDir+'visualization')
plot_metadata(cistopic_obj,
                 reduction_name = 'UMAP',
                 variables=['scenic_ident', 'pycisTopic_leiden_10_0.6'], 
                 target='cell', 
                 num_columns=2,
                 text_size=10,
                 dot_size=5,
                 figsize=(20,5),
                 save= outDir + 'visualization/merged_dimensionality_reduction_label.pdf')

In [ ]:
plot_topic(cistopic_obj,
            reduction_name = 'UMAP',
            target = 'cell',
            num_columns=3)

In [ ]:
cell_topic_heatmap(cistopic_obj,
                     variables = ['clust.names', 'scenic_ident'],
                     scale = False,
                     legend_loc_x = 1.05,
                     legend_loc_y = -1.2,
                     legend_dist_y = -1,
                     figsize=(10,10))

In [ ]:
with open(outDir + 'merged_CT_object.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

In [ ]:
#os.mkdir(outDir+'topic_binarization')
from pycisTopic.topic_binarization import *
region_bin_topics = binarize_topics(cistopic_obj, method='otsu', ntop=3000, plot=True, num_columns=5, save= outDir + 'topic_binarization/merged_otsu.pdf')

In [ ]:
binarized_cell_topic = binarize_topics(cistopic_obj, target='cell', method='li', plot=True, num_columns=5, nbins=60)

In [ ]:
from pycisTopic.topic_qc import *
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

In [ ]:
topic_qc_metrics

In [ ]:
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

In [ ]:
import matplotlib

In [ ]:
# Plot topic stats in one figure

fig=plt.figure(figsize=(40, 43))
i = 1
for fig_ in fig_dict.keys():
    plt.subplot(2, 3, i)
    img = pycisTopic.utils.fig2img(fig_dict[fig_]) #To convert figures to png to plot together, see .utils.py. This converts the figure to png.
    plt.imshow(img)
    plt.axis('off')
    i += 1
plt.subplots_adjust(wspace=0, hspace=-0.70)
fig.savefig(outDir + 'topic_binarization/Topic_merged_qc.pdf', bbox_inches='tight')
plt.show()

# example of a good topic: it has a high coherence and a high number of assignmnents/marginal topic score (these measurements are correlated),
#while being cell type specific (with a high gini index). 
#Generally, we will consider a topic as bad if it has a low number of assignments while being assigned to many cells and a low coherence.

In [ ]:
topic_annot = topic_annotation(cistopic_obj, annot_var = 'scenic_ident', binarized_cell_topic = binarized_cell_topic, general_topic_thr = 0.2)

In [ ]:
topic_annot

In [ ]:
topic_qc_metrics = pd.concat([topic_annot[['scenic_ident', 'Ratio_cells_in_topic', 'Ratio_group_in_population']], topic_qc_metrics], axis=1)

In [ ]:
topic_qc_metrics

In [ ]:
# Save
with open(outDir + 'topic_binarization/merged_Topic_qc_metrics_annot.pkl', 'wb') as f:
  pickle.dump(topic_qc_metrics, f)
with open(outDir + 'topic_binarization/merged_binarized_cell_topic.pkl', 'wb') as f:
  pickle.dump(binarized_cell_topic, f)
with open(outDir + 'topic_binarization/merged_binarized_topic_region.pkl', 'wb') as f:
  pickle.dump(region_bin_topics, f)

In [ ]:
from pycisTopic.diff_features import *
imputed_acc_obj = impute_accessibility(cistopic_obj, selected_cells=None, selected_regions=None, scale_factor=10**6)

In [ ]:
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
#os.mkdir(outDir+'DARs')
variable_regions = find_highly_variable_features(normalized_imputed_acc_obj,
                                           min_disp = 0.05,
                                           min_mean = 0.0125, 
                                           max_mean = 3,
                                           max_disp = np.inf,
                                           n_bins = 20, 
                                           n_top_features = None,
                                           plot = True)

In [ ]:
len(variable_regions)

In [ ]:
markers_dict = find_diff_features(cistopic_obj, 
                      imputed_acc_obj,
                      variable = 'scenic_ident',
                      var_features = variable_regions,
                      contrasts = None,
                      adjpval_thr = 0.05,
                      log2fc_thr = np.log2(1.5),
                      n_cpu = 5,
                      split_pattern = '-') 

In [ ]:
from pycisTopic.clust_vis import *
plot_imputed_features(cistopic_obj,
                    reduction_name = 'UMAP',
                    imputed_data = imputed_acc_obj,
                    features = [markers_dict[x].index.tolist()[0] for x in ['FLHSC', 'BMHSC', 'FLLy', 'BMLy']],
                    scale = False,
                    num_columns = 4)

In [ ]:
x = [print(x + ': '+ str(len(markers_dict[x]))) for x in markers_dict.keys()]

In [ ]:
markers_dict.keys()

In [ ]:
[markers_dict[x].index.tolist()[0] for x in ['FLHSC', 'BMHSC', 'FLLy', 'BMLy']] 

In [ ]:
with open(outDir + 'candidate_enhancers/merged_markers_dict_cluster.pkl', 'wb') as f:
  pickle.dump(markers_dict, f)

In [ ]:
# Get TSS annotations
import pybiomart as pbm
import pyranges as pr
dataset = pbm.Dataset(name = 'hsapiens_gene_ensembl',  host = 'http://www.ensembl.org')
annot = dataset.query(attributes=['chromosome_name', 'start_position', 'end_position', 'strand', 'external_gene_name', 'transcription_start_site', 'transcript_biotype'])
annot['Chromosome/scaffold name'] = 'chr' + annot['Chromosome/scaffold name'].astype(str)
annot.columns=['Chromosome', 'Start', 'End', 'Strand', 'Gene','Transcription_Start_Site', 'Transcript_type']
annot = annot[annot.Transcript_type == 'protein_coding']
annot.Strand[annot.Strand == 1] = '+'
annot.Strand[annot.Strand == -1] = '-'
pr_annotation = pr.PyRanges(annot.dropna(axis = 0))

In [ ]:
# Get chromosome sizes
import requests

chromsizes=pd.read_csv(projDir + 'fasta/hg38.chrom.sizes', sep='\t', header=None)
chromsizes.columns=['Chromosome', 'End']
chromsizes['Start']=[0]*chromsizes.shape[0]
chromsizes=chromsizes.loc[:,['Chromosome', 'Start', 'End']]
chromsizes=pr.PyRanges(chromsizes)

# save chromosome sizes for snakemake pipeline
chromsizes.to_csv(projDir + 'outs/chromsizes.tsv', sep='\t')

In [ ]:
# # save chromosome sizes for scenicplus snakemake pipeline
# chromsizes=pd.read_csv(projDir + 'fasta/hg38.chrom.sizes', sep='\t', header=None)
# chromsizes.columns=['Chromosome', 'End']
# chromsizes['Start']=[0]*chromsizes.shape[0]
# chromsizes=chromsizes.loc[:,['Chromosome', 'Start', 'End']]

# # chromsizes.to_csv('/home/spalo/scenicplus/outs/chromsizes.tsv', sep='\t')

In [ ]:
from pycisTopic.gene_activity import *
gene_act, weigths = get_gene_activity(imputed_acc_obj, # Region-cell probabilities
                pr_annotation, # Gene annotation
                chromsizes, # Chromosome size
                use_gene_boundaries=True, # Whether to use the whole search space or stop when encountering another gene
                upstream=[1000, 100000], # Search space upstream. The minimum means that even if there is a gene right next to it 
                                      #these bp will be taken (1kbp here)
                downstream=[1000,100000], # Search space downstream
                distance_weight=True, # Whether to add a distance weight (an exponential function, the weight will decrease with distance)
                decay_rate=1, # Exponent for the distance exponential funciton (the higher the faster will be the decrease)
                extend_gene_body_upstream=10000, # Number of bp upstream immune to the distance weight (their value will be maximum for 
                                      #this weight)
                extend_gene_body_downstream=500, # Number of bp downstream immune to the distance weight
                gene_size_weight=False, # Whether to add a weights based on the length of the gene
                gene_size_scale_factor='median', # Dividend to calculate the gene size weigth. Default is the median value of all genes
                                      #in the genome
                remove_promoters=False, # Whether to remove promoters when computing gene activity scores
                average_scores=True, # Whether to divide by the total number of region assigned to a gene when calculating the gene 
                                      #activity score
                scale_factor=1, # Value to multiply for the final gene activity matrix
                extend_tss=[10,10], # Space to consider a promoter
                gini_weight = True, # Whether to add a gini index weigth. The more unique the region is, the higher this weight will be
                return_weights= True, # Whether to return the final weights
                project='Gene_activity') # Project name for the gene activity object

In [ ]:
DAG_markers_dict = find_diff_features(cistopic_obj, 
                      gene_act,
                      variable='scenic_ident',
                      var_features=None,
                      contrasts=None,
                      adjpval_thr=0.05,
                      log2fc_thr=np.log2(1.5),
                      n_cpu=5
                      )  

In [ ]:
with open(outDir + 'candidate_enhancers/merged_markers_dict_DAGs.pkl', 'wb') as f:
  pickle.dump(DAG_markers_dict, f)

In [ ]:
from pycisTopic.clust_vis import *
plot_imputed_features(cistopic_obj,
                    reduction_name='UMAP',
                    imputed_data=gene_act,
                    features=['HOXB4', 'NFE2', 'EBF1', 'MPO'], 
                    scale=True,
                    num_columns=4)

In [ ]:
with open(outDir + 'merged_CT_object.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

In [ ]:
region_bin_topics['Topic1'].index

In [ ]:
#import pyranges as pr
from pycistarget.utils import *
region_sets = {key: pr.PyRanges(region_names_to_coordinates(region_bin_topics[key].index.tolist())) for key in region_bin_topics.keys()}

In [ ]:
region_sets

In [ ]:
#make directories for saving output
os.makedirs(os.path.join(outDir, "region_sets_merged"), exist_ok = True)
os.makedirs(os.path.join(outDir, "region_sets_merged", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(outDir, "region_sets_merged", "DARs_cell_type"), exist_ok = True)

In [ ]:
from pycisTopic.utils import region_names_to_coordinates

In [ ]:
for topic in region_bin_topics:
    region_names_to_coordinates(
        region_bin_topics[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outDir, "region_sets_merged", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outDir, "region_sets_merged", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    ) #results in an empty bed file in the DARs folder for FL_atac_HSC, i.e. the non-cluster 0 HSCs, that needs to be removed before 
#running the snakemake pipeline

In [ ]:
# try importing the CITE scanpy object
import scanpy as sc

In [ ]:
cite = sc.read_h5ad(projDir + 'merged/merged_CITE.h5ad')
cite

In [ ]:
print(cite.X)

In [ ]:
cite.raw = cite.copy()
sc.pp.normalize_total(cite, target_sum=1e4)
sc.pp.log1p(cite)

In [ ]:
print(cite.raw.X)

In [ ]:
print(cite.X)

In [ ]:
cite.obs['scenic_ident']

In [ ]:
#read metadata and change cell names to match counts matrix
cell_data = pd.read_csv(projDir + 'merged/merged_CITE_md.tsv', sep = '\t')
cell_data.index = cell_data.index.str.replace("[-]", ".", regex = True)

cell_data.head()

In [ ]:
cite.obs['scenic_ident'][:] = cell_data['scenic_ident'][:]

In [ ]:
cite.obs['scenic_ident'] = cite.obs['scenic_ident'].astype('category')

In [ ]:
cite.obs['scenic_ident']

In [ ]:
cite.write_h5ad(projDir + 'merged/merged_CITE_edit.h5ad') 